# Week 4: L1 vs L2 on PLS scores

Does L1 logistic on the 32 PLS-DA scores zero out everything except $t_1$ (and maybe $t_2$)?

Same balanced pure-cell LORO folds as the PLS notebook. Fold rotations come from `outputs/week4_pls_loro/checkpoints/`. Training: torch proximal L1 on GPU, mean BCE plus $(1/C)\|w\|_1$. Ran on 8 GPUs / 80 CPUs in 79.4 s.

```bash
.venv/bin/python -u scripts/train_week4_l1_pls_loro.py --backend gpu
.venv/bin/python -u scripts/build_week4_l1_pls_notebook.py
```

## Result

L1 does sparsify. As $C$ drops (stronger penalty), mean nonzero count falls 32 → ~16 ($C=100$) → ~6 ($C=30$) → ~1 ($C=10$, almost always $t_1$). Stronger than that zeros the head.

| Head | $C$ | mean nnz | LORO macro-F1 |
|---|---:|---:|---:|
| L2 on PLS $t_{1..32}$ | — | 32.0 | 0.775 ± 0.088 |
| L2 on AE 64-D | — | 64.0 | 0.779 ± 0.085 |
| L1 on PLS (~$t_1$ only) | 10.0 | 1.20 | 0.735 |

Share of folds where each score stays nonzero ($|\beta_k| > 10^{-6}$):

| $C$ | mean nnz | $t_1$ | $t_2$ | $t_3$ | $t_4$ | $t_5$ | $t_6$ |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 10 | 1.2 | 1.00 | 0.20 | 0.00 | 0.00 | 0.00 | 0.00 |
| 30 | 6.3 | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 |
| 100 | 15.9 | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 |


## L1 equation

Scores are standardized inside each fold: $z_k = (t_k - m_k)/s_k$. Numbers below average the LORO fold fits.

### $C=10$: one score

Mean nnz 1.2. $t_1$ appears in every fold; $t_2$ in two (Atlanta, LA), with a small weight.

$$
\tilde{x}_j = \frac{A_j - \mu_j}{\sigma_j},
\quad
t_1 = \mathbf{r}_1^\top \tilde{\mathbf{x}},
\quad
z_1 = \frac{t_1}{s_1}\ (s_1 \approx 3.17)
$$

$$
\mathrm{logit} = 0.011 + 0.962\, z_1 = 0.011 + 0.304\, t_1
$$

$$
P(\mathrm{RES}\mid\mathbf{x}) = \sigma(\mathrm{logit})
$$

LORO macro-F1 ≈ 0.735.

| Metro | intercept | $t_1$ weight | $t_2$ weight |
|---|---:|---:|---:|
| Atlanta_SE | +0.011 | 0.96 | 0.01 |
| Boston_NE | +0.015 | 1.00 | — |
| Chicago_MW | +0.010 | 1.01 | — |
| Denver_W | +0.010 | 0.92 | — |
| Houston_S | +0.013 | 0.97 | — |
| LA_W | +0.006 | 0.85 | 0.09 |
| NYC_NE | +0.009 | 1.03 | — |
| Phoenix_SW | +0.011 | 0.93 | — |
| SF_W | +0.010 | 0.98 | — |
| Seattle_NW | +0.010 | 0.96 | — |

![t1 weights by metro](../../outputs/week4_l1_pls_loro/figures/l1_c10_t1_coefs_by_region.png)

### $C=30$: six scores

Mean nnz 6.3. $t_1$ through $t_6$ stay nonzero in essentially every fold.

$$
\begin{aligned}
\mathrm{logit} \approx\ &0.029 \\
&+ 1.395\, z_1 + 0.337\, z_2 + 0.221\, z_3 \\
&+ 0.105\, z_4 + 0.088\, z_5 + 0.036\, z_6
\end{aligned}
$$

On raw scores ($z_k \approx t_k / s_k$):

$$
\mathrm{logit} \approx 0.029 + 0.440\, t_1 + 0.160\, t_2 + 0.105\, t_3 + 0.066\, t_4 + 0.069\, t_5 + 0.026\, t_6
$$

LORO macro-F1 ≈ 0.751 (+0.016 vs $C=10$).


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("../..").resolve()
OUT = ROOT / "outputs" / "week4_l1_pls_loro"
summary = json.loads((OUT / "summary.json").read_text())
coefs = pd.read_csv(OUT / "l1_c10_fold_coefs.csv")
print("C=10 survival:", summary.get("survival_C10_t1_to_t10"))
coefs.sort_values("region")


![L1 path on PLS scores](../../outputs/week4_l1_pls_loro/figures/l1_path_pls32.png)

![L1 path on AE bands](../../outputs/week4_l1_pls_loro/figures/l1_path_ae64.png)

![Which PLS scores survive](../../outputs/week4_l1_pls_loro/figures/l1_survival_pls32.png)

![|coef| vs C for early scores](../../outputs/week4_l1_pls_loro/figures/l1_coef_magnitudes_pls.png)


## Full L1 path (PLS scores)

| penalty | $C$ | mean nnz | LORO F1 |
|---|---:|---:|---:|
| L1 | 0.0010 | 0.0 | 0.333 ± 0.000 |
| L1 | 0.0030 | 0.0 | 0.333 ± 0.000 |
| L1 | 0.0100 | 0.0 | 0.333 ± 0.000 |
| L1 | 0.0300 | 0.0 | 0.333 ± 0.000 |
| L1 | 0.1000 | 0.0 | 0.333 ± 0.000 |
| L1 | 0.3000 | 0.0 | 0.333 ± 0.000 |
| L1 | 1.0000 | 0.0 | 0.333 ± 0.000 |
| L1 | 3.0000 | 0.0 | 0.333 ± 0.000 |
| L1 | 10.0000 | 1.2 | 0.735 ± 0.119 |
| L1 | 30.0000 | 6.3 | 0.751 ± 0.105 |
| L1 | 100.0000 | 15.9 | 0.763 ± 0.094 |
| L2 | — | 32.0 | 0.775 ± 0.088 |


## Load sweep tables


In [ ]:
sweep = pd.read_csv(OUT / "sweep_summary.csv")
metrics = pd.read_csv(OUT / "loro_metrics.csv")
sweep.loc[sweep.features == "pls32"].sort_values(["penalty", "C"])


### Surviving names per fold at $C=10$


In [ ]:
metrics.loc[
    (metrics.features == "pls32") & (metrics.penalty == "l1") & (metrics.C == 10.0),
    ["region", "nnz", "macro_f1", "surviving"]
].sort_values("region")


### Surviving names per fold at $C=30$


In [ ]:
metrics.loc[
    (metrics.features == "pls32") & (metrics.penalty == "l1") & (metrics.C == 30.0),
    ["region", "nnz", "macro_f1", "surviving"]
].sort_values("region")


## Takeaways

1. L2 keeps all 32 PLS scores (nnz 32, F1 ≈ 0.775).
2. L1 sparsifies as $C$ falls: nnz 32 → ~16 ($C=100$) → ~6 ($C=30$) → ~1 ($C=10$) → 0 ($C \le 3$).
3. At $C=10$ the head is $\mathrm{logit} = 0.011 + 0.304\, t_1$, F1 ≈ 0.735.
4. At $C=30$ you still keep $t_1\ldots t_6$; “only $t_1$ and $t_2$” is the strong end of the path, not the whole path.
5. L1 on raw AE also zeros bands, but those axes are not orthogonal. Sparse weights on $t_k$ are easier to read.

Files: `outputs/week4_l1_pls_loro/` (see also `l1_equation.md`, `l1_c10_fold_coefs.csv`).
